In [ ]:
import trimesh
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# Load files
file_paths = {
    'OBJ': '/content/dog.obj',
    'STL': '/content/duck.stl',
    'GLTF': '/content/scene.gltf'
}

# Load models
meshes = {}
for fmt, path in file_paths.items():
    try:
        mesh = trimesh.load(path, force='mesh')
        meshes[fmt] = mesh
    except Exception as e:
        print(f"Error loading {fmt}: {e}")


print(f"{'Format':<10} {'Vertices':<10} {'Faces':<10} {'Normals':<10} {'Duplicates':<10}")
for fmt, mesh in meshes.items():
    n_vertices = len(mesh.vertices)
    n_faces = len(mesh.faces)
    has_normals = hasattr(mesh, 'vertex_normals') and mesh.vertex_normals.shape[0] == mesh.vertices.shape[0]

    # Has vertex

    def count_duplicate_vertices(mesh):
      verts = mesh.vertices
      unique_verts = np.unique(verts, axis=0)
      num_duplicates = len(verts) - len(unique_verts)
      return num_duplicates

    n_duplicates = count_duplicate_vertices(mesh)

    print(f"{fmt:<10} {n_vertices:<10} {n_faces:<10} {str(has_normals):<10} {n_duplicates:<10}")

# File trimesh viewer
# meshes['OBJ'].show()  # or 'STL', 'GLTF'

# Open3D viewer
def view_open3d(path):
    mesh_o3d = o3d.io.read_triangle_mesh(path)
    mesh_o3d.compute_vertex_normals()
    o3d.visualization.draw_geometries([mesh_o3d])

# view_open3d(file_paths['OBJ'])  -> Jupyter

# Save OBJ as STL and GLTF
mesh = meshes['OBJ']
mesh.export('converted_model_stl.stl')
mesh.export('converted_model_glft.gltf')
mesh.export('converted_model_glb.glb')  # or .stl, .gltf, .ply

def load_model(path):
    mesh = trimesh.load_mesh(path)
    vertices = mesh.vertices
    faces = mesh.faces
    # Delete double edges
    unique_edges = set(frozenset(edge) for edge in mesh.edges)
    edges = np.array([list(e) for e in unique_edges])
    return vertices, edges, faces

def plot_model(vertices, faces, color):
    fig = plt.figure()
    fig.set_size_inches(10, 10)
    ax = fig.add_subplot(projection='3d', box_aspect=(1, 1, 1))

    mesh = Poly3DCollection(vertices[faces], alpha=0.5)
    mesh.set_facecolor(color)
    ax.add_collection3d(mesh)

    scale = vertices.flatten()
    ax.auto_scale_xyz(scale, scale, scale)

    plt.show()

models = [
    ("dog.obj", "#964B00"),
    ("converted_model_stl.stl", "#8A9597"),
    ("converted_model_glb.glb", "#EFB810"),
    ("duck.stl", "#339933"),
    ("cat.glb", "#FFC0CB"),
]

for path, color in models:
    vertices, edges, faces = load_model(path)
    plot_model(vertices, faces, color)